# Does a small model know what it doesn't know?

Calibration study of an open language model, 4-bit, on a single RTX 4070 (12 GB).

**This notebook currently covers Stage 1, step a1 (protocol §0-§1):** load the model in 4-bit and run one
forward pass. Confirm it fits in VRAM and that the logits tensor prints. Nothing else matters until this works.

Later steps (a2 dataset, a3 scoring function, a4 `scores.csv`, ...) are added below as they are built.

## §0 Environment check

The GPU must be visible to PyTorch, and `bitsandbytes` must be built against the right CUDA.

In [ ]:
import torch

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "No CUDA device visible - install the cu124 build of torch (protocol §0)"

props = torch.cuda.get_device_properties(0)
print("GPU            :", props.name)
print("VRAM total     :", round(props.total_memory / 1e9, 1), "GB")

import transformers, bitsandbytes
print("transformers   :", transformers.__version__)
print("bitsandbytes   :", bitsandbytes.__version__)

## §1 Load the model in 4-bit

`MODEL_ID` is the only thing that changes between the two model variants. Everything else in the pipeline
stays identical, so any difference between conditions is not caused by loading.

- **stock**: `Qwen/Qwen3-8B`
- **uncensored**: an abliterated Qwen3-8B checkpoint from the Hugging Face Hub, *same base*, loaded with the
  identical `BitsAndBytesConfig`. The exact repo id is not chosen yet; it is needed before step c4.

This loads full-precision Hugging Face weights and quantises them on the fly to nf4. GGUF files are **not**
used here: the protocol needs raw logits from `transformers`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

VARIANT  = "stock"            # "stock" | "uncensored"
MODEL_IDS = {
    "stock": "Qwen/Qwen3-8B",
    "uncensored": None,       # TODO before step c4: abliterated Qwen3-8B repo id (or local HF-format folder)
}
MODEL_ID = MODEL_IDS[VARIANT]
assert MODEL_ID, f"No model id set for variant {VARIANT!r}"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="cuda:0",
)
model.eval()

print(MODEL_ID)
print("model footprint:", round(model.get_memory_footprint() / 1e9, 2), "GB   (expect roughly 5-6 GB for an 8B model at nf4)")

## Step a1: one forward pass

A single hard-coded question, run through the model's chat template, then one forward pass. The point is only to
confirm the model runs on the GPU and that we can read the logits. `enable_thinking=False` stops Qwen3 opening a
`<think>` block; if your installed `transformers` does not support it, this is where it will show up.

In [ ]:
question = "Which of these is a source of light?"
options  = ["The Moon", "A mirror", "The Sun", "A window"]

body = "\n".join(f"{L}. {t}" for L, t in zip("ABCD", options))
user = ("Answer the multiple-choice question with a single letter.\n\n"
        f"Question: {question}\n{body}")

prompt = tok.apply_chat_template(
    [{"role": "user", "content": user}],
    tokenize=False, add_generation_prompt=True,
    enable_thinking=False,
) + "Answer:"

print(prompt)

In [ ]:
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**inputs)

logits = out.logits                      # shape: [batch, sequence, vocab]
print("logits tensor shape :", tuple(logits.shape))
print("last-position logits:", logits[0, -1])

# What does the model want to say next? (a quick look, before any scoring function exists)
probs = torch.softmax(logits[0, -1].float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{p:6.3f}  id={i:<7} {tok.convert_ids_to_tokens(i)!r}")

In [ ]:
peak = torch.cuda.max_memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"peak VRAM allocated: {peak:.2f} GB of {total:.1f} GB")
assert peak < total * 0.95, "Too close to the VRAM limit - consider a 4B model (protocol §1)"
print("a1 done: model loads in 4-bit, forward pass runs, logits print.")

## What to check before moving on to a2

- The footprint is roughly 5-6 GB and the peak VRAM is comfortably under 12 GB.
- The top next-token candidates look like an answer (a letter, likely with a leading space), not `<think>` or
  garbage. If you see `<think>`, upgrade `transformers` or append the template's empty-think marker manually.
- Note anything that broke (version drift in `transformers` / `bitsandbytes` is common) in `notes/breakage-log.md`.